# Looking at the MTBS fire data

Just checking out the fire records, where fires happened, how big they were, and whether any data is missing.

## 1. Import stuff

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# Find the main project folder so the data paths still work from explorations/
PROJECT_ROOT = next((p for base in (Path.cwd(), Path.cwd().resolve()) for p in [base, *base.parents] if (p / "data" / "raw").is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not find data/raw. Put explorations/ inside your fire-prediction project.")
RAW = PROJECT_ROOT / "data" / "raw"
print("Project root:", PROJECT_ROOT)
print("Data folder:", RAW)
import geopandas as gpd

## 2. What MTBS files do I have?

In [ ]:
mtbs_files = sorted(p for p in RAW.rglob('*') if p.is_file() and ('mtbs' in str(p.relative_to(RAW)).lower()))
display(pd.DataFrame([{'file':str(p.relative_to(RAW)), 'MB':round(p.stat().st_size/1e6,2)} for p in mtbs_files]))

## 3. Load the fire data

In [ ]:
shp = RAW / "mtbs" / "mtbs_perims_DD.shp"
gpkg = RAW / "mtbs_2020_2020.gpkg"

if shp.exists():
    fires = gpd.read_file(shp)
    source = shp
elif gpkg.exists():
    fires = gpd.read_file(gpkg)
    source = gpkg
else:
    raise FileNotFoundError("No MTBS shapefile or GeoPackage found under data/raw")

print("Source:", source.relative_to(PROJECT_ROOT))
print("Number of fire records:", len(fires))
print("Coordinate reference system (CRS):", fires.crs)
print("Geometry types:", fires.geometry.geom_type.value_counts().to_dict())
print("Columns:", fires.columns.tolist())
display(fires.head())

## 4. Look for missing data

In [ ]:
display(pd.DataFrame({"dtype": fires.dtypes.astype(str),
                      "missing": fires.isna().sum(),
                      "missing_pct": (fires.isna().mean() * 100).round(1)}))
display(fires.describe(include="all").transpose().head(40))

## 5. Plot the fires on a map

In [ ]:
# Get coordinates ready for plotting
plot_fires = fires.to_crs("EPSG:4326") if fires.crs else fires
fig, ax = plt.subplots(figsize=(12, 8))
plot_fires.plot(ax=ax, color="orangered", edgecolor="darkred", linewidth=0.25, alpha=0.5)
ax.set_title("MTBS wildfire perimeters")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
plt.tight_layout()
plt.show()

## 6. Which fires were the biggest?

I am estimating the area from the shapes here, so the numbers might not be exactly the same as the reported burned acreage.

In [ ]:
if fires.crs is None:
    print("This layer does not declare a CRS; skip area calculations until that is resolved.")
else:
    area_fires = fires.to_crs("EPSG:5070").copy()  # use this to calculate area in acres
    area_fires["area_acres_est"] = area_fires.geometry.area / 4046.8564224
    display(area_fires.drop(columns="geometry").sort_values("area_acres_est", ascending=False).head(10))
    fig, ax = plt.subplots(figsize=(9, 4))
    area_fires["area_acres_est"].dropna().plot.hist(bins=40, ax=ax)
    ax.set_xlabel("Fire polygon area (approx. acres)")
    ax.set_title("Fire perimeter area distribution")
    plt.tight_layout()
    plt.show()

## 7. Check the 2020 MTBS file

This looks at the different layers in the GeoPackage file.

In [ ]:
if gpkg.exists():
    print("GeoPackage layers:")
    try:
        import fiona
        layers = fiona.listlayers(gpkg)
    except (ImportError, AttributeError):
        import pyogrio
        layers = pyogrio.list_layers(gpkg)[:, 0].tolist()
    print(layers)
    for layer in layers:
        sample = gpd.read_file(gpkg, layer=layer, rows=5)
        print(f"\nLayer: {layer} | CRS: {sample.crs} | columns: {sample.columns.tolist()}")
        display(sample.head())
else:
    print("No GeoPackage found")

## 8. Things to keep in mind

MTBS does not include every single fire. Also, fire size and perimeter are things we know after a fire, so they would not work as inputs for predicting whether a fire starts.